<a href="https://colab.research.google.com/github/Raffy0-1/DHC-ML-Task_5/blob/main/Failed_Mental_Health_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install transformers datasets accelerate

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("atharvjairath/empathetic-dialogues-facebook-ai")

print("Path to dataset files:", path)

100%|██████████| 3.26M/3.26M [00:00<00:00, 5.05MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/atharvjairath/empathetic-dialogues-facebook-ai/versions/1


In [ ]:
import os

print(os.listdir(path))

['emotion-emotion_69k.csv']


In [ ]:
import pandas as pd

df = pd.read_csv(path + "/emotion-emotion_69k.csv")
df.head()

,Unnamed: 0,Situation,emotion,empathetic_dialogues,labels,Unnamed: 5,Unnamed: 6
0,0,I remember going to the fireworks with my best...,sentimental,Customer :I remember going to see the firework...,"Was this a friend you were in love with, or ju...",NaN,NaN
1,1,I remember going to the fireworks with my best...,sentimental,Customer :This was a best friend. I miss her.\...,Where has she gone?,NaN,NaN
2,2,I remember going to the fireworks with my best...,sentimental,Customer :We no longer talk.\nAgent :,Oh was this something that happened because of...,NaN,NaN
3,3,I remember going to the fireworks with my best...,sentimental,Customer :Was this a friend you were in love w...,This was a best friend. I miss her.,NaN,NaN
4,4,I remember going to the fireworks with my best...,sentimental,Customer :Where has she gone?\nAgent :,We no longer talk.,NaN,NaN


In [ ]:
df.columns

Index(['Unnamed: 0', 'Situation', 'emotion', 'empathetic_dialogues', 'labels',
       'Unnamed: 5', 'Unnamed: 6'],
      dtype='object')

In [ ]:
# Strip column names just in case
df.columns = df.columns.str.strip()

# Keep only relevant mental-health-focused emotions
relevant_emotions = [
    'sad', 'lonely', 'afraid', 'terrified', 'anxious',
    'devastated', 'apprehensive', 'ashamed', 'guilty', 'disappointed'
]

df_filtered = df[df['emotion'].isin(relevant_emotions)].copy()
df_filtered.shape

(19645, 7)

In [ ]:
df_filtered['final_input'] = "Emotion: " + df_filtered['emotion'] + "\nCustomer: " + df_filtered['empathetic_dialogues']
df_filtered['final_output'] = "Agent: " + df_filtered['labels']

df_filtered[['final_input', 'final_output']].head()

,final_input,final_output
5,Emotion: afraid\nCustomer: Customer : it feels...,Agent: Oh ya? I don't really see how
6,Emotion: afraid\nCustomer: Customer :dont you ...,Agent: I do actually hit blank walls a lot of ...
7,Emotion: afraid\nCustomer: Customer : i virtua...,Agent: Wait what are sweatings
8,Emotion: afraid\nCustomer: Customer :Oh ya? I ...,Agent: dont you feel so.. its a wonder
9,Emotion: afraid\nCustomer: Customer :I do actu...,Agent: i virtually thought so.. and i used to...


In [ ]:
df_filtered.to_csv("preprocessed_mental_health.csv", index=False)

In [ ]:
for i in range(5):
    print(df_filtered['final_input'].iloc[i])
    print(df_filtered['final_output'].iloc[i])
    print("----")

Emotion: afraid
Customer: Customer : it feels like hitting to blank wall when i see the darkness
Agent :
Agent: Oh ya? I don't really see how
----
Emotion: afraid
Customer: Customer :dont you feel so.. its a wonder 
Agent :
Agent: I do actually hit blank walls a lot of times but i get by
----
Emotion: afraid
Customer: Customer : i virtually thought so.. and i used to get sweatings
Agent :
Agent: Wait what are sweatings
----
Emotion: afraid
Customer: Customer :Oh ya? I don't really see how
Agent :
Agent: dont you feel so.. its a wonder 
----
Emotion: afraid
Customer: Customer :I do actually hit blank walls a lot of times but i get by
Agent :
Agent:  i virtually thought so.. and i used to get sweatings
----


In [ ]:
# remove the extra 'customer'
df_filtered['final_input'] = df_filtered['final_input'].str.replace("Customer: Customer :", "Customer:", regex=False)

In [ ]:
# Remove extra 'Agent :' at the end of input
df_filtered['final_input'] = df_filtered['final_input'].str.replace(r"Agent :$", "", regex=True).str.strip()

In [ ]:
# Strip whitespace
df_filtered['final_output'] = df_filtered['final_output'].str.strip()

# Remove rows where output is empty
df_filtered = df_filtered[df_filtered['final_output'] != ""]
df_filtered = df_filtered.dropna(subset=['final_output'])

In [ ]:
for i in range(5):
    print("Input:", df_filtered['final_input'].iloc[i])
    print("Output:", df_filtered['final_output'].iloc[i])
    print("----")

Input: Emotion: afraid
Customer: it feels like hitting to blank wall when i see the darkness
Output: Agent: Oh ya? I don't really see how
----
Input: Emotion: afraid
Customer:dont you feel so.. its a wonder
Output: Agent: I do actually hit blank walls a lot of times but i get by
----
Input: Emotion: afraid
Customer: i virtually thought so.. and i used to get sweatings
Output: Agent: Wait what are sweatings
----
Input: Emotion: afraid
Customer:Oh ya? I don't really see how
Output: Agent: dont you feel so.. its a wonder
----
Input: Emotion: afraid
Customer:I do actually hit blank walls a lot of times but i get by
Output: Agent:  i virtually thought so.. and i used to get sweatings
----


In [ ]:
df_filtered.to_csv("preprocessed_mental_health.csv", index=False)

# Importing Libraries

AutoTokenizer → converts text → numbers (model understands numbers only)

AutoModelForCausalLM → the actual GPT model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, DataCollatorForLanguageModeling, Trainer
from datasets import Dataset

In [ ]:
df_filtered = pd.read_csv("preprocessed_mental_health.csv")

In [ ]:
hf_dataset = Dataset.from_pandas(df_filtered)

In [ ]:
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
max_length = 128

def tokenize_function(examples):
    full_texts = [inp + "\n" + out for inp, out in
                  zip(examples["final_input"], examples["final_output"])]

    tokenized = tokenizer(
        full_texts,
        max_length=max_length,
        truncation=True,
        padding="max_length"
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/19645 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./mental_health_model",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    logging_steps=50,
    save_steps=200,
    eval_strategy="steps",       # ✅ evaluate every N steps
    eval_steps=200,              # ✅ evaluate every 200 steps
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    save_total_limit=2,
    fp16=True,
    push_to_hub=False,
    load_best_model_at_end=True,  # ✅ keeps best checkpoint
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # False = causal LM (GPT-style), not masked LM
)

In [ ]:
# Split the dataset into training and evaluation sets
split_dataset = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,  # Pass the evaluation dataset
    data_collator=data_collator
)

In [ ]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
200,2.882431,2.745981
400,2.856481,2.718034
600,2.744854,2.689058
800,2.802604,2.667170
1000,2.696865,2.647981
1200,2.699463,2.633438
1400,2.687264,2.613918
1600,2.640934,2.606201
1800,2.647503,2.592740
2000,2.624964,2.583744


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=13260, training_loss=2.4348106961084888, metrics={'train_runtime': 4410.8881, 'train_samples_per_second': 12.025, 'train_steps_per_second': 3.006, 'total_flos': 1732397456424960.0, 'train_loss': 2.4348106961084888, 'epoch': 3.0})

In [ ]:
trainer.save_model("./mental_health_model_final")
tokenizer.save_pretrained("./mental_health_model_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./mental_health_model_final/tokenizer_config.json',
 './mental_health_model_final/tokenizer.json')

In [ ]:
from transformers import pipeline, GenerationConfig

chat_model = AutoModelForCausalLM.from_pretrained("./mental_health_model_final")
chat_tokenizer = AutoTokenizer.from_pretrained("./mental_health_model_final")

# Removed explicit GenerationConfig object creation

chat_pipeline = pipeline(
    "text-generation",
    model=chat_model,
    tokenizer=chat_tokenizer,
    device=0  # use GPU; set to -1 if CPU only
    # Removed generation_config=generation_config
)

def detect_emotion(text):
    text = text.lower()
    if any(w in text for w in ["angry", "anger", "furious", "mad"]):
        return "angry"
    elif any(w in text for w in ["sad", "cry", "depressed", "hopeless"]):
        return "sad"
    elif any(w in text for w in ["anxious", "anxiety", "nervous", "worried"]):
        return "anxious"
    elif any(w in text for w in ["lonely", "alone", "isolated"]):
        return "lonely"
    elif any(w in text for w in ["scared", "afraid", "terrified", "fear"]):
        return "afraid"
    else:
        return "sad"  # default

def chat():
    print("Mental Health Chatbot (type 'quit' to exit)")
    while True:
        user_input = input("You: ")
        if user_input.lower() == "quit":
            break

        emotion = detect_emotion(user_input)
        prompt = f"Emotion: {emotion}\nCustomer: {user_input}\nAgent:"

        response = chat_pipeline(
            prompt,
            max_new_tokens=60,  # Adjusted to encourage more concise responses
            do_sample=True,
            top_k=50,
            top_p=0.9,
            temperature=0.7, # Slightly reduced temperature for more coherent output
            repetition_penalty=1.2,
            pad_token_id=chat_tokenizer.eos_token_id
        )

        generated_text = response[0]['generated_text']

        # Extract only what comes after "Agent:" and strip leading/trailing whitespace
        bot_reply = generated_text.split("Agent:")[-1].strip()

        # Remove any partial prompt elements that might have been generated
        # Split by "Emotion:" and "Customer:" to ensure these are not part of the reply
        bot_reply = bot_reply.split("Emotion:")[0]
        bot_reply = bot_reply.split("Customer:")[0].strip()

        # Remove any leading/trailing Agent: or Agent : from the reply
        if bot_reply.lower().startswith("agent:"):
            bot_reply = bot_reply[len("agent:"):].strip()
        if bot_reply.lower().startswith("agent "):
            bot_reply = bot_reply[len("agent "):].strip()

        print(f"Bot [{emotion}]: {bot_reply}\n")

chat()

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Mental Health Chatbot (type 'quit' to exit)
You: i am feeling bad


Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot [sad]: I hope so.  Im very sorry to hear that!  I hope everything is okay though, and you're all right :) You are all right. Thank you for the kind words! Just be confident! Hopefully your health will improve. And that it's not a big deal. But hopefully that



KeyboardInterrupt: Interrupted by user